# MuseTalk 1.5 GPU validation worker

Test worker only. Uses an approved traditional-clothing singer image/video and successful ACE-Step bhajan audio, then runs MuseTalk 1.5 on a Colab GPU and produces a lip-synced MP4.

Enable a T4 GPU. Do not publish until visual QA confirms identity, traditional clothing, devotional presentation and actual lip-sync.

In [ ]:
# Runtime and repository setup
!nvidia-smi
!pip -q install uv
import subprocess
from pathlib import Path
MT=Path('/content/MuseTalk')
VENV=Path('/content/musetalk310')
if not MT.exists():
    subprocess.run(['git','clone','--depth','1','https://github.com/TMElyralab/MuseTalk.git',str(MT)],check=True)
if not (VENV/'bin/python').exists():
    subprocess.run(['uv','venv','--python','3.10',str(VENV)],check=True)
PY=str(VENV/'bin/python')
print('MuseTalk:',MT)
subprocess.run([PY,'-V'],check=True)


In [ ]:
# Install dependencies inside Python 3.10. Do not use ensurepip.
import subprocess
PY='/content/musetalk310/bin/python'
MT='/content/MuseTalk'
subprocess.run(['uv','pip','install','--python',PY,'pip','setuptools','wheel'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'torch==2.0.1','torchvision==0.15.2','torchaudio==2.0.2','--index-url','https://download.pytorch.org/whl/cu118'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'-r',MT+'/requirements.txt'],check=True)
subprocess.run(['uv','pip','install','--python',PY,'openmim'],check=True)
MIM='/content/musetalk310/bin/mim'
for pkg in ['mmengine','mmcv==2.0.1','mmdet==3.1.0','mmpose==1.1.0']:
    subprocess.run([MIM,'install',pkg],check=True)
print('MuseTalk dependencies installed.')


In [ ]:
# Download official model/component weights
import os,subprocess
os.chdir('/content/MuseTalk')
subprocess.run(['bash','./download_weights.sh'],check=True)
print('All weights downloaded successfully.')


In [ ]:
# Upload the approved singer image/video and the successful ACE-Step audio.
from google.colab import files
from pathlib import Path
print('Upload the APPROVED traditional-clothing singer image/video:')
uploaded_avatar=files.upload()
avatar=next(iter(uploaded_avatar))
print('Upload the successful ACE-Step bhajan MP3/WAV:')
uploaded_audio=files.upload()
audio=next(iter(uploaded_audio))
assert Path(avatar).suffix.lower() in {'.png','.jpg','.jpeg','.webp','.mp4','.mov','.webm'}, f'Unsupported avatar: {avatar}'
assert Path(audio).suffix.lower() in {'.mp3','.wav','.m4a','.flac','.aac','.ogg'}, f'Please upload the actual audio file, not a ZIP: {audio}'
print('Avatar:',avatar)
print('Audio:',audio)


In [ ]:
# Normalize inputs. MuseTalk supports an image directly, so keep the approved singer image as the source.
import subprocess
from pathlib import Path
OUT=Path('/content/musetalk_output')
OUT.mkdir(exist_ok=True)
avatar_src=OUT/'avatar_source.png'
audio_wav=OUT/'audio.wav'
subprocess.run(['ffmpeg','-y','-i',avatar,'-frames:v','1','-vf','scale=512:-2','-pix_fmt','rgb24',str(avatar_src)],check=True)
subprocess.run(['ffmpeg','-y','-i',audio,'-ar','16000','-ac','1',str(audio_wav)],check=True)
assert avatar_src.exists() and avatar_src.stat().st_size>10000
assert audio_wav.exists() and audio_wav.stat().st_size>10000
print('Avatar:',avatar_src)
print('Audio:',audio_wav)


In [ ]:
# Create the actual MuseTalk task configuration.
from pathlib import Path
MT=Path('/content/MuseTalk')
OUT=Path('/content/musetalk_output')
cfg=MT/'configs/inference/test.yaml'
cfg.write_text(f'''bhajan_test:
  video_path: "{OUT/'avatar_source.png'}"
  audio_path: "{OUT/'audio.wav'}"
  result_name: "bhajan_lipsync.mp4"
''')
print(cfg.read_text())


In [ ]:
# MuseTalk 1.5 inference
import os,subprocess
os.chdir('/content/MuseTalk')
cmd=['/content/musetalk310/bin/python','-m','scripts.inference','--inference_config','configs/inference/test.yaml','--result_dir','/content/musetalk_output/result','--unet_model_path','models/musetalkV15/unet.pth','--unet_config','models/musetalkV15/musetalk.json','--whisper_dir','models/whisper','--version','v15','--fps','25','--batch_size','4','--use_float16','--parsing_mode','jaw']
print('Starting MuseTalk 1.5 on T4...')
subprocess.run(cmd,check=True)
print('MUSE TALK FINISHED')


In [ ]:
# Locate and download the final MP4
from pathlib import Path
from google.colab import files
result_dir=Path('/content/musetalk_output/result')
candidates=list(result_dir.rglob('*.mp4'))
assert candidates,'MuseTalk produced no MP4'
candidate=max(candidates,key=lambda p:p.stat().st_size)
assert candidate.stat().st_size>100000,f'Output MP4 is unexpectedly small: {candidate.stat().st_size} bytes'
print('SUCCESS:',candidate)
print('Size:',round(candidate.stat().st_size/1024/1024,2),'MB')
files.download(str(candidate))
